# Notebook 1 — Fine-tune FinBERT for Financial Sentiment (Pipeline 1)
**ISOM5240 Group Project — Earnings Call Dividend Risk Early Warning System**

This notebook fine-tunes `ProsusAI/finbert` on the Financial PhraseBank dataset
for 3-class sentiment classification: `positive`, `negative`, `neutral`.

The fine-tuned model is pushed to HuggingFace Hub and used as Pipeline 1
in the Streamlit app to score each transcript chunk for financial negativity.

**Steps:**
1. Install packages
2. Load & explore Financial PhraseBank dataset
3. Tokenize and preprocess
4. Fine-tune FinBERT with HuggingFace Trainer
5. Evaluate accuracy and F1
6. Test inference on sample sentences
7. Push fine-tuned model to HuggingFace Hub


In [16]:
# ── Step 1: Install required packages ──────────────────────────────────────
!pip install -q transformers datasets accelerate scikit-learn huggingface_hub
!pip install -q sentencepiece

In [17]:
# ── Step 2: Imports ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    pipeline,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report
import torch
print('PyTorch version:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cu128
GPU available: True
GPU: Tesla T4


In [18]:
# ── Step 3: HuggingFace login (enter your token when prompted) ──────────────
from huggingface_hub import notebook_login
notebook_login()

# Set your HuggingFace username here
HF_USERNAME = 'ruirui0506'            # ← CHANGE THIS
MODEL_NAME   = 'finbert-dividend-sentiment'
FULL_MODEL_ID = f'{HF_USERNAME}/{MODEL_NAME}'

In [19]:
# ── Step 4: Load Financial PhraseBank dataset (Fixed – no legacy script error) ──
# Download the original zip (public mirror) and load the highest-agreement version

!wget -q https://github.com/neoyipeng2018/FinancialPhraseBank-v1.0/raw/main/FinancialPhraseBank-v1.0.zip -O FinancialPhraseBank-v1.0.zip

import zipfile
import pandas as pd
from datasets import Dataset

# Extract
with zipfile.ZipFile('FinancialPhraseBank-v1.0.zip', 'r') as z:
    z.extractall()

# Load Sentences_AllAgree.txt (2264 samples, 100% annotator agreement)
df = pd.read_csv('FinancialPhraseBank-v1.0/Sentences_AllAgree.txt',
                 sep='@',
                 header=None,
                 names=['sentence', 'label'],
                 encoding='iso-8859-1')

print("Dataset loaded successfully!")
print(df.head(3))
print('\nLabel distribution:')
print(df['label'].value_counts())

raw = Dataset.from_pandas(df)
raw = raw.train_test_split(test_size=0.2, seed=42)
# ── Encode string labels → integers (required for PyTorch training) ──────────
LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}

df['label'] = df['label'].str.strip().map(LABEL2ID)

# Verify no NaN (would mean an unexpected label string was found)
assert df['label'].isna().sum() == 0, "Unmapped labels found — check raw file encoding"

print('\nLabel distribution (encoded):')
print(df['label'].value_counts().sort_index().rename(ID2LABEL))

raw = Dataset.from_pandas(df)
raw = raw.train_test_split(test_size=0.2, seed=42)
print('\nFinal dataset structure:')
print(raw)
print('\nFinal dataset structure:')
print(raw)

Dataset loaded successfully!
                                            sentence     label
0  According to Gran , the company has no plans t...   neutral
1  For the last quarter of 2010 , Componenta 's n...  positive
2  In the third quarter of 2010 , net sales incre...  positive

Label distribution:
label
neutral     1391
positive     570
negative     303
Name: count, dtype: int64

Label distribution (encoded):
label
negative     303
neutral     1391
positive     570
Name: count, dtype: int64

Final dataset structure:
DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 1811
    })
    test: Dataset({
        features: ['sentence', 'label'],
        num_rows: 453
    })
})

Final dataset structure:
DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 1811
    })
    test: Dataset({
        features: ['sentence', 'label'],
        num_rows: 453
    })
})


In [20]:
# ── Step 5: Explore class distribution ─────────────────────────────────────
from collections import Counter
labels = raw['train']['label']
dist = Counter(labels)
label_names = [ID2LABEL[i] for i in sorted(ID2LABEL)]    # ['negative','neutral','positive']
print('Class distribution:')
for k, v in sorted(dist.items()):
    print(f'  {label_names[k]:10s}: {v}')
print(f'\nTotal samples: {len(labels)}')

Class distribution:
  negative  : 230
  neutral   : 1111
  positive  : 470

Total samples: 1811


In [21]:
# ── Step 6: Train / Validation / Test split ─────────────────────────────────
# Financial PhraseBank only has a 'train' split — we create val and test splits.
# Using sklearn for stratified splitting (more robust than HF's built-in stratify).
from sklearn.model_selection import train_test_split
import pandas as pd

df_all = raw['train'].to_pandas()
train_df, temp_df = train_test_split(
    df_all, test_size=0.2, stratify=df_all['label'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42
)

from datasets import Dataset, DatasetDict
dataset = DatasetDict({
    'train':      Dataset.from_pandas(train_df, preserve_index=False),
    'validation': Dataset.from_pandas(val_df,   preserve_index=False),
    'test':       Dataset.from_pandas(test_df,  preserve_index=False),
})
print('Split sizes:')
for split, ds in dataset.items():
    print(f'  {split:12s}: {len(ds)} samples')

Split sizes:
  train       : 1448 samples
  validation  : 181 samples
  test        : 182 samples


In [22]:
# ── Step 7: Tokenize ────────────────────────────────────────────────────────
BASE_MODEL = 'ProsusAI/finbert'
tokenizer  = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize_fn(examples):
    return tokenizer(
        examples['sentence'],
        truncation=True,
        padding='max_length',
        max_length=128,     # FinPhraseBank sentences are short
    )

tokenized = dataset.map(tokenize_fn, batched=True)
tokenized = tokenized.rename_column('label', 'labels')
tokenized.set_format('torch', columns=['input_ids','attention_mask','labels'])
print('Tokenization complete.')
print('Sample input_ids shape:', tokenized['train'][0]['input_ids'].shape)

Map:   0%|          | 0/1448 [00:01<?, ? examples/s]

Map:   0%|          | 0/181 [00:00<?, ? examples/s]

Map:   0%|          | 0/182 [00:00<?, ? examples/s]

Tokenization complete.
Sample input_ids shape: torch.Size([128])


In [23]:
# ── Step 8: Load pre-trained FinBERT and replace head ───────────────────────
NUM_LABELS = 3   # negative=0, neutral=1, positive=2

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=NUM_LABELS,
    id2label={0:'negative', 1:'neutral', 2:'positive'},
    label2id={'negative':0, 'neutral':1, 'positive':2},
    ignore_mismatched_sizes=True,
)
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:    {total_params:,}')
print(f'Trainable params:{trainable:,}')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total params:    109,484,547
Trainable params:109,484,547


In [24]:
# ── Step 9: Define evaluation metrics ──────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc   = accuracy_score(labels, preds)
    f1    = f1_score(labels, preds, average='weighted')
    return {'accuracy': acc, 'f1': f1}

In [27]:
# ── Step 10: Training arguments ─────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir                  = f'./{MODEL_NAME}',
    num_train_epochs            = 5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    lr_scheduler_type           = 'cosine',
    eval_strategy               = 'epoch',   # ← renamed from evaluation_strategy
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    greater_is_better           = True,
    logging_steps               = 50,
    fp16                        = torch.cuda.is_available(),
    report_to                   = 'none',
    push_to_hub                 = True,
    hub_model_id                = FULL_MODEL_ID,
)
print('Training arguments set.')

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training arguments set.


In [29]:
# ── Step 11: Instantiate Trainer and train ──────────────────────────────────
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = tokenized['train'],
    eval_dataset    = tokenized['validation'],
    processing_class  = tokenizer,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)
print('Starting training ...')
train_result = trainer.train()
print('\nTraining complete!')
print(f'Training runtime: {train_result.metrics["train_runtime"]:.1f} s')
print(f'Samples/second:   {train_result.metrics["train_samples_per_second"]:.1f}')

Starting training ...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.977248,0.254607,0.911602,0.908647
2,0.120497,0.146857,0.955801,0.955398
3,0.029239,0.094034,0.977901,0.977901
4,0.020503,0.091543,0.977901,0.977901
5,0.006545,0.088708,0.977901,0.977901


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Training complete!
Training runtime: 296.0 s
Samples/second:   24.5


In [30]:
# ── Step 12: Evaluate on test set ───────────────────────────────────────────
test_results = trainer.evaluate(tokenized['test'])
print('\n── Test Set Results ──────────────────────────────')
for k, v in test_results.items():
    print(f'  {k:30s}: {v:.4f}')

# Detailed classification report
preds_out  = trainer.predict(tokenized['test'])
y_pred     = np.argmax(preds_out.predictions, axis=-1)
y_true     = preds_out.label_ids
print('\n── Classification Report ─────────────────────────')
print(classification_report(y_true, y_pred, target_names=['negative','neutral','positive']))


── Test Set Results ──────────────────────────────
  eval_loss                     : 0.0664
  eval_accuracy                 : 0.9890
  eval_f1                       : 0.9889
  eval_runtime                  : 0.4049
  eval_samples_per_second       : 449.4780
  eval_steps_per_second         : 14.8180
  epoch                         : 5.0000

── Classification Report ─────────────────────────
              precision    recall  f1-score   support

    negative       1.00      0.91      0.95        23
     neutral       1.00      1.00      1.00       112
    positive       0.96      1.00      0.98        47

    accuracy                           0.99       182
   macro avg       0.99      0.97      0.98       182
weighted avg       0.99      0.99      0.99       182



In [31]:
# ── Step 13: Test inference on sample sentences ─────────────────────────────
sample_sentences = [
    'The company reported record-breaking revenue growth and raised its dividend by 10%.',
    'Management is reviewing the sustainability of the current payout ratio.',
    'Free cash flow turned negative; the company may need to suspend its dividend.',
    'We expect stable earnings in line with analyst consensus estimates.',
    'Debt covenants were breached; lenders have issued a waiver pending restructuring.',
]

sent_pipeline = pipeline(
    'text-classification', model=trainer.model,
    tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1,
    top_k=None,
)

print('Inference results:')
for sent in sample_sentences:
    res = sent_pipeline(sent[:512])
    scores = {item['label']: f"{item['score']:.3f}" for item in res[0]}
    print(f'  [{max(res[0], key=lambda x: x["score"])["label"].upper():8s}] {sent[:60]}...')
    print(f'           Scores: {scores}')

Inference results:
  [POSITIVE] The company reported record-breaking revenue growth and rais...
           Scores: {'positive': '0.995', 'negative': '0.003', 'neutral': '0.002'}
  [NEUTRAL ] Management is reviewing the sustainability of the current pa...
           Scores: {'neutral': '0.996', 'negative': '0.003', 'positive': '0.002'}
  [NEGATIVE] Free cash flow turned negative; the company may need to susp...
           Scores: {'negative': '0.970', 'neutral': '0.021', 'positive': '0.009'}
  [POSITIVE] We expect stable earnings in line with analyst consensus est...
           Scores: {'positive': '0.988', 'negative': '0.009', 'neutral': '0.002'}
  [NEUTRAL ] Debt covenants were breached; lenders have issued a waiver p...
           Scores: {'neutral': '0.841', 'negative': '0.155', 'positive': '0.004'}


In [32]:
# ── Step 14: Save and push model to HuggingFace Hub ─────────────────────────
trainer.save_model(f'./{MODEL_NAME}')
tokenizer.save_pretrained(f'./{MODEL_NAME}')
trainer.push_to_hub(commit_message='Fine-tuned FinBERT for dividend sentiment analysis')
print(f'\n✅ Model pushed to: https://huggingface.co/{FULL_MODEL_ID}')
print('   Use this ID in app.py → SENTIMENT_MODEL_ID')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ntiment/training_args.bin: 100%|##########| 5.26kB / 5.26kB            

...

New Data Upload               : |          |  0.00B /  0.00B            

  ...ntiment/training_args.bin: 100%|##########| 5.26kB / 5.26kB            

  ...ntiment/model.safetensors:  27%|##7       |  120MB /  438MB            

No files have been modified since last commit. Skipping to prevent empty commit.



✅ Model pushed to: https://huggingface.co/ruirui0506/finbert-dividend-sentiment
   Use this ID in app.py → SENTIMENT_MODEL_ID
